# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library, following the FAIR principles and Croissant schema.

### Dataset Source
The dataset schema is available at:
**[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)**


In [ ]:
# Install the mlcroissant library (if necessary)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, associated fields and columns.

All references to dataset entities (record sets, fields, columns) use their `@id` for robust referencing. Let's list available record sets and their fields.

In [ ]:
# List all record sets by their @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name       : {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print('')

For demonstration, let's print a few records from one primary record set. **Be sure to replace `<record_set_id>` with the chosen record set `@id`**.

In [ ]:
# Pick the first available record set for exploration (update @id as needed)
if record_sets:
    main_record_set = record_sets[0]
    main_record_set_id = main_record_set.id
    print(f"Sample records from record set @id: {main_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:  # Print only a few
            break
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load all available record sets into Pandas DataFrames, referenced by their `@id`s.

In [ ]:
# Create DataFrames for each record set
dataframes = {}
for rs in record_sets:
    print(f"Extracting records for RecordSet @id={rs.id} ...")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample data (first 2 rows):\n{df.head(2)}\n")

# For further analysis, pick the main record set
record_set_id = main_record_set_id
print(f"Columns in main record set dataframe (@id={record_set_id}):\n{dataframes[record_set_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Let's process a numeric field, such as protein abundance or molecular weight. **Pick any numeric field (by `@id`)—for demonstration, we select the first numeric column present in the chosen record set.**

We'll filter the records, normalize the selected numeric column, and (if possible) group by a meaningful field.

In [ ]:
# Identify a suitable numeric field by data type (float or int)
main_fields = {field.id: field for field in main_record_set.fields}
df = dataframes[record_set_id]

numeric_field_id = None
for field in main_record_set.fields:
    if field.data_type in ['Number', 'Float', 'Integer']:
        if field.id in df.columns:
            numeric_field_id = field.id
            print(f"Selected numeric field @id: {numeric_field_id} ({field.name})")
            break
if not numeric_field_id:
    raise ValueError('No numeric field found for EDA.')

# Drop NA for clean numeric operations
df_num = df.dropna(subset=[numeric_field_id])

# Examine value distribution
print(f"Value statistics for {numeric_field_id}:")
print(df_num[numeric_field_id].describe())

# Filter records where the numeric value exceeds its median
threshold = df_num[numeric_field_id].median()
filtered_df = df_num[df_num[numeric_field_id] > threshold]
print(f"Filtered records ({numeric_field_id} > median [{threshold}]):")
print(filtered_df.head(3))

# Normalize the field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Try grouping by a string/categorical column (search for category candidate)
group_field_id = None
for field in main_record_set.fields:
    if field.data_type in ['Text', 'String'] and field.id in df.columns:
        group_field_id = field.id
        print(f"Grouping by field @id: {group_field_id} ({field.name})")
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data (mean {numeric_field_id}) by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the numeric field (histogram) and (if available) the average values by a grouping variable.

In [ ]:
# Visualization requires matplotlib/seaborn
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_num[numeric_field_id], kde=True)
plt.title(f"Distribution of field {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping available, barplot of means
if group_field_id:
    plt.figure(figsize=(7,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Average {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- This notebook demonstrates loading a Croissant-compliant dataset using `mlcroissant` via its schema URL.
- All dataset entities are referenced by their `@id` (record sets, fields, etc.) for precise, future-proof analysis.
- Initial exploration included a summary of the record sets, fields, and normalization of a numeric field, concluding with basic visualizations.

Further analysis can build on these steps for more detailed biological or proteomics-specific insights.